### 📂 Day 15: Automated RAG Evaluation with RAGAS

**The Goal:** Use an LLM to "audit" your RAG pipeline and provide metrics for **Faithfulness** (no hallucinations) and **Answer Relevancy**.

#### 1. The Core Metrics

RAGAS looks at the relationship between the **Question**, the **Retrieved Context**, and the **Generated Answer**.

* **Faithfulness:** Does the answer stay 100% within the retrieved context? (Checks for hallucinations).
* **Answer Relevancy:** Does the answer actually address the user's specific query?
* **Context Precision:** Did the system retrieve the *right* information from the database?

#### 2. The Code Logic (Colab Implementation)

You will need to install `ragas` and `datasets`. We will simulate a small evaluation dataset based on your college subjects.

In [21]:
# 1. Install necessary libraries
!pip install -q langchain-groq ragas datasets langchain-community sentence-transformers langchain-huggingface

import os
from ragas import evaluate
from datasets import Dataset
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

# 2. Initialize your Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key="GROQ_API_KEY"
)

# 3. Initialize HuggingFace Embeddings (Free/Local)
# This avoids the OpenAI requirement for embedding the dataset
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 4. Initialize metrics and link them to our Groq LLM
faithfulness = Faithfulness(llm=llm)
answer_relevancy = AnswerRelevancy(llm=llm)
context_precision = ContextPrecision(llm=llm)

# 5. Sample Data
data_samples = {
    'question': ['What is the Blackwell architecture?', 'How does NVLink impact AI scaling?'],
    'answer': [
        'Blackwell is NVIDIA\'s latest GPU architecture designed for generative AI.',
        'NVLink allows multiple GPUs to communicate at high speeds.'
    ],
    'contexts' : [
        ['NVIDIA Blackwell platform features 208 billion transistors for generative AI.'],
        ['The fifth-generation NVLink delivers 1.8TB/s bidirectional throughput per GPU.']
    ],
    'ground_truth': [
        'Blackwell is NVIDIA\'s new GPU architecture.',
        'NVLink is a high-speed interconnect for multi-GPU scaling.'
    ]
}
dataset = Dataset.from_dict(data_samples)

# 6. Run the Evaluation explicitly passing BOTH the LLM and Embeddings
print("🚀 Starting Ragas Evaluation using Groq as the Judge and HF for Embeddings...")
result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=llm,
    embeddings=embeddings
)

# 7. Final Results - FIXED
df = result.to_pandas()

print("\n🔍 Debug: Available columns in the result:", df.columns.tolist())

# Ragas 0.2+ usually names them like this:
# 'user_input' (instead of question), 'faithfulness', 'answer_relevance', 'context_precision'

# Let's use a flexible print that won't crash:
cols_to_show = [col for col in ['user_input', 'question', 'faithfulness', 'answer_relevance', 'answer_relevancy', 'context_precision'] if col in df.columns]

print("\n📊 RAG Evaluation Results:")
print(df[cols_to_show])

/tmp/ipykernel_52282/1595877438.py:9: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
/tmp/ipykernel_52282/1595877438.py:9: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision
/tmp/ipykernel_52282/1595877438.py:9: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Starting Ragas Evaluation using Groq as the Judge and HF for Embeddings...


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


🔍 Debug: Available columns in the result: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision']

📊 RAG Evaluation Results:
                            user_input  faithfulness  answer_relevancy  \
0  What is the Blackwell architecture?           0.5          0.818897   
1   How does NVLink impact AI scaling?           1.0          0.574149   

   context_precision  
0                1.0  
1                1.0  
